# Fase 5.2 — Fine-tuning LoRA sobre WikiArt

Entrena adaptadores LoRA en el UNet de cada modelo sobre el corpus WikiArt.
El VAE y el encoder de texto permanecen congelados.

Ejecutar **una vez por modelo**. Cambiar `MODEL_KEY` y ejecutar todo.

**Prerequisito:** `fase5_dataset.ipynb` completado.

---
## 0 — Entorno

In [ ]:
from google.colab import drive
import os, sys, json, time, math
import torch
import torch.nn.functional as F

drive.mount("/content/drive", force_remount=False)
PROJECT_ROOT = "/content/drive/MyDrive/TFM/repositorio"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["HF_HOME"] = "/content/.cache/huggingface"

assert torch.cuda.is_available(), "GPU no disponible. Activa T4."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = ""  # <-- pega tu token hf_...

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN vacio. Rellena la variable con tu token de "
        "https://huggingface.co/settings/tokens (tipo Read) "
        "y vuelve a ejecutar esta celda."
    )

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
print("Autenticado en Hugging Face.")

In [ ]:
req = os.path.join(PROJECT_ROOT, "requirements_colab.txt")
!pip install -q -U -r {req}
import diffusers, peft, transformers
print(f"diffusers={diffusers.__version__}  peft={peft.__version__}  transformers={transformers.__version__}")

---
## 1 — Configuracion

> **Unico parametro a cambiar entre sesiones:** `MODEL_KEY`

In [ ]:
# Configuración de fine-tuning LoRA. Los hiperparámetros son idénticos para
# los tres modelos salvo donde la VRAM lo impide. SDXL usa menos pasos (750
# vs 1000) y batch más pequeño (1 vs 2) por su mayor huella de memoria.
# grad_accum=4 mantiene un batch efectivo de 4-8 imágenes, suficiente para
# estabilizar el gradiente con 500 imágenes de entrenamiento.
# gradient_checkpointing en SDXL intercambia cómputo por memoria (~30% más
# lento pero hace viable el entrenamiento dentro de los 15 GB de la T4).
# ── CAMBIAR ESTO ENTRE SESIONES ──────────────────────────────────────────────
MODEL_KEY = "sd15"   # "sd15" | "sd21" | "sdxl"
# ─────────────────────────────────────────────────────────────────────────────

MODEL_CONFIGS = {
    "sd15": {
        "model_id": "runwayml/stable-diffusion-v1-5",
        "is_sdxl": False, "resolution": 512,
        "train_steps": 1000, "batch_size": 2, "grad_accum": 4, "grad_checkpoint": False,
    },
    "sd21": {
        "model_id": "stabilityai/stable-diffusion-2-1",
        "is_sdxl": False, "resolution": 768,
        "train_steps": 1000, "batch_size": 2, "grad_accum": 4, "grad_checkpoint": False,
    },
    "sdxl": {
        "model_id": "stabilityai/stable-diffusion-xl-base-1.0",
        "is_sdxl": True, "resolution": 1024,
        "train_steps": 750, "batch_size": 1, "grad_accum": 4, "grad_checkpoint": True,
    },
}

LORA_R       = 8
LORA_ALPHA   = 8
LORA_DROPOUT = 0.1
LORA_MODULES = ["to_q", "to_k", "to_v", "to_out.0"]
LR           = 1e-4
LR_WARMUP    = 100
WEIGHT_DECAY = 1e-2

cfg          = MODEL_CONFIGS[MODEL_KEY]
FINETUNE_DIR = os.path.join(PROJECT_ROOT, "data", "finetune")
OUT_DIR      = os.path.join(FINETUNE_DIR, f"lora_{MODEL_KEY}")
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Modelo         : {MODEL_KEY}  ({cfg['model_id']})")
print(f"Resolucion     : {cfg['resolution']}x{cfg['resolution']}")
print(f"Pasos          : {cfg['train_steps']}  |  Batch efectivo: {cfg['batch_size']*cfg['grad_accum']}")
print(f"LoRA r={LORA_R}  alpha={LORA_ALPHA}  modulos={LORA_MODULES}")
print(f"Salida         : {OUT_DIR}")

---
## 2 — Dataset WikiArt

In [ ]:
# Dataset PyTorch para WikiArt. El pipeline de transformaciones aplica
# RandomHorizontalFlip para aumentar la diversidad efectiva del corpus,
# y normaliza a [-1, 1] (rango esperado por el VAE de Diffusers).
# Para SDXL se tokeniza el caption con ambos tokenizadores (CLIP ViT-L y
# OpenCLIP ViT-bigG) porque el UNet requiere ambas representaciones textuales.
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

manifest_path = os.path.join(FINETUNE_DIR, "manifest.json")
images_dir    = os.path.join(FINETUNE_DIR, "images")
assert os.path.exists(manifest_path), (
    f"manifest.json no encontrado: {manifest_path}\n"
    "Ejecuta primero fase5_dataset.ipynb"
)
with open(manifest_path) as f:
    manifest_data = json.load(f)

class WikiArtDataset(Dataset):
    def __init__(self, manifest, images_dir, resolution, tokenizer,
                 is_sdxl=False, tokenizer_2=None):
        self.manifest    = manifest
        self.images_dir  = images_dir
        self.tokenizer   = tokenizer
        self.tokenizer_2 = tokenizer_2
        self.is_sdxl     = is_sdxl
        self.tf = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.LANCZOS),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.manifest)

    def _tok(self, tok, text):
        return tok(text, max_length=tok.model_max_length,
                   padding="max_length", truncation=True,
                   return_tensors="pt").input_ids.squeeze(0)

    def __getitem__(self, idx):
        e   = self.manifest[idx]
        img = Image.open(os.path.join(self.images_dir, e["filename"])).convert("RGB")
        item = {"pixel_values": self.tf(img),
                "input_ids":    self._tok(self.tokenizer, e["caption"])}
        if self.is_sdxl:
            if self.tokenizer_2 is None:
                raise ValueError(
                    "WikiArtDataset: is_sdxl=True pero tokenizer_2 es None. "
                    "Asegurate de ejecutar la celda de carga del modelo antes del bucle."
                )
            item["input_ids_2"] = self._tok(self.tokenizer_2, e["caption"])
        return item

print(f"Dataset WikiArt: {len(manifest_data)} imagenes")

---
## 3 — Carga del modelo y configuracion LoRA

El VAE y el encoder de texto se congelan; solo los adaptadores LoRA del UNet son entrenables.

In [ ]:
# Diseño experimental clave: se congelan VAE y encoder(es) de texto.
# La congelación del VAE garantiza que z₀_pre y z₀_ft son comparables en el
# mismo espacio vectorial, haciendo interpretable la diferencia de latentes.
# La congelación del encoder de texto asegura que cualquier cambio en z₀ es
# atribuible exclusivamente a la reorganización del proceso de denoising del
# UNet, no a cambios en el espacio de condicionamiento textual.
# Solo los adaptadores LoRA del UNet son entrenables (~0.4% de los parámetros).
from diffusers import StableDiffusionPipeline, StableDiffusionXLPipeline
from peft import LoraConfig, get_peft_model

device = torch.device("cuda")

print(f"Cargando {cfg['model_id']} ...")
if cfg["is_sdxl"]:
    pipe = StableDiffusionXLPipeline.from_pretrained(
        cfg["model_id"], torch_dtype=torch.float16, use_safetensors=True)
else:
    pipe = StableDiffusionPipeline.from_pretrained(
        cfg["model_id"], torch_dtype=torch.float16, use_safetensors=True)

vae            = pipe.vae.to(device).eval()
text_encoder   = pipe.text_encoder.to(device).eval()
text_encoder_2 = getattr(pipe, "text_encoder_2", None)
if text_encoder_2:
    text_encoder_2 = text_encoder_2.to(device).eval()
tokenizer   = pipe.tokenizer
tokenizer_2 = getattr(pipe, "tokenizer_2", None)

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
if text_encoder_2:
    text_encoder_2.requires_grad_(False)

unet = pipe.unet.to(device)
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_MODULES, bias="none", init_lora_weights=True,
)
unet = get_peft_model(unet, lora_config)
if cfg["grad_checkpoint"]:
    unet.enable_gradient_checkpointing()
unet.train()

# SDXL: VAE y text_encoder_2 en fp32 — evitan NaN en encoding y proyeccion CLIP-bigG
if cfg["is_sdxl"]:
    vae.to(dtype=torch.float32)
    if text_encoder_2 is not None:
        text_encoder_2.to(dtype=torch.float32)
    print("VAE + text_encoder_2 → fp32 (estabilidad numerica SDXL)")

# LoRA adapters a fp32: el optimizer actualiza en fp32, autocast hace el forward en fp16
for param in unet.parameters():
    if param.requires_grad:
        param.data = param.data.float()

unet.print_trainable_parameters()
print(f"VRAM usada: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

---
## 4 — Bucle de entrenamiento

In [ ]:
# Bucle de entrenamiento con mixed precision (fp16 autocast + GradScaler).
# El objetivo de denoising es predecir el ruido añadido (epsilon-prediction)
# o la velocidad del flujo (v_prediction para SD 2.1). El scheduler DDPM
# gestiona el nivel de ruido en cada timestep t muestreado uniformemente.
# Se usa gradient clipping (max_norm=1.0) para estabilizar el entrenamiento
# con LoRA, cuyos gradientes pueden ser grandes en las primeras iteraciones.
# El learning rate sigue un schedule coseno con warm-up de 100 pasos.
from diffusers import DDPMScheduler
from torch.optim.lr_scheduler import LambdaLR

noise_sched = DDPMScheduler.from_pretrained(cfg["model_id"], subfolder="scheduler")
pred_type   = noise_sched.config.prediction_type
print(f"Tipo de prediccion: {pred_type}")

if cfg["is_sdxl"] and tokenizer_2 is None:
    raise RuntimeError(
        "MODEL_KEY='sdxl' pero tokenizer_2 es None. "
        "Ejecuta primero la celda '3 — Carga del modelo' con MODEL_KEY='sdxl'."
    )

dataset = WikiArtDataset(
    manifest_data, images_dir, cfg["resolution"],
    tokenizer, cfg["is_sdxl"], tokenizer_2,
)
loader = DataLoader(
    dataset, batch_size=cfg["batch_size"], shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)

optimizer = torch.optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY,
)
total_steps = cfg["train_steps"]
res         = cfg["resolution"]

def lr_lambda(step):
    if step < LR_WARMUP:
        return step / max(1, LR_WARMUP)
    prog = (step - LR_WARMUP) / max(1, total_steps - LR_WARMUP)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * prog)))

lr_sched    = LambdaLR(optimizer, lr_lambda)
scaler      = torch.cuda.amp.GradScaler()
global_step = 0
accum_count = 0
step_loss   = 0.0
train_log   = []
t0          = time.time()
optimizer.zero_grad()

print(f"Entrenando {total_steps} pasos (batch efectivo={cfg['batch_size']*cfg['grad_accum']})...")

while global_step < total_steps:
    for batch in loader:
        if global_step >= total_steps:
            break

        pv = batch["pixel_values"].to(device, dtype=torch.float16)

        with torch.no_grad():
            encode_in = pv.float() if cfg["is_sdxl"] else pv
            latents = vae.encode(encode_in).latent_dist.sample() * vae.config.scaling_factor
            if cfg["is_sdxl"]:
                latents = latents.to(dtype=torch.float16)
            bsz     = latents.shape[0]
            noise   = torch.randn_like(latents)
            t       = torch.randint(
                0, noise_sched.config.num_train_timesteps, (bsz,), device=device).long()
            noisy   = noise_sched.add_noise(latents, noise, t)

            ids = batch["input_ids"].to(device)
            if cfg["is_sdxl"]:
                ids2   = batch["input_ids_2"].to(device)
                hs1    = text_encoder(ids,  output_hidden_states=True).hidden_states[-2]
                out2   = text_encoder_2(ids2, output_hidden_states=True)
                hs2    = out2.hidden_states[-2]
                pooled = out2[0]
                enc    = torch.cat([hs1, hs2], dim=-1)
                time_ids = torch.tensor(
                    [[res, res, 0, 0, res, res]],
                    device=device, dtype=torch.float16).repeat(bsz, 1)
                added_cond = {"text_embeds": pooled, "time_ids": time_ids}
            else:
                enc = text_encoder(ids).last_hidden_state

            target = (noise_sched.get_velocity(latents, noise, t)
                      if pred_type == "v_prediction" else noise)

        with torch.cuda.amp.autocast(dtype=torch.float16):
            if cfg["is_sdxl"]:
                pred = unet(noisy, t, encoder_hidden_states=enc,
                            added_cond_kwargs=added_cond).sample
            else:
                pred = unet(noisy, t, encoder_hidden_states=enc).sample
            loss = F.mse_loss(pred.float(), target.float(), reduction="mean")

        scaler.scale(loss / cfg["grad_accum"]).backward()
        step_loss   += loss.item()
        accum_count += 1

        if accum_count % cfg["grad_accum"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in unet.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            lr_sched.step()
            optimizer.zero_grad()
            global_step += 1

            avg_loss  = step_loss / cfg["grad_accum"]
            step_loss = 0.0

            if global_step % 50 == 0 or global_step == 1:
                elapsed = (time.time() - t0) / 60
                print(f"  Paso {global_step:>4}/{total_steps}  "
                      f"loss={avg_loss:.5f}  lr={lr_sched.get_last_lr()[0]:.2e}  {elapsed:.1f}min")
                train_log.append({"step": global_step, "loss": avg_loss})

            if global_step >= total_steps:
                break

print(f"\nCompletado: {global_step} pasos en {(time.time()-t0)/60:.1f} min")


---
## 5 — Guardado de los adaptadores LoRA

In [ ]:
unet.save_pretrained(OUT_DIR)

log_path = os.path.join(OUT_DIR, "training_log.json")
with open(log_path, "w") as f:
    json.dump({
        "model": MODEL_KEY, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "lora_modules": LORA_MODULES, "train_steps": global_step,
        "batch_size": cfg["batch_size"], "grad_accum": cfg["grad_accum"],
        "lr": LR, "total_time_min": (time.time() - t0) / 60,
        "log": train_log,
    }, f, indent=2)

adapter_files = [fn for fn in os.listdir(OUT_DIR) if fn.endswith((".safetensors", ".bin"))]
adapter_mb = sum(os.path.getsize(os.path.join(OUT_DIR, fn)) for fn in adapter_files) / 1024**2
print(f"Adaptador LoRA : {OUT_DIR}")
print(f"Archivos       : {adapter_files}")
print(f"Tamano         : {adapter_mb:.1f} MB")

---
## 6 — Curva de perdida

In [ ]:
import matplotlib.pyplot as plt

steps  = [e["step"] for e in train_log]
losses = [e["loss"] for e in train_log]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, color="#4C72B0", lw=2)
plt.xlabel("Paso"); plt.ylabel("MSE loss")
plt.title(f"{MODEL_KEY.upper()} — Curva de perdida LoRA (WikiArt)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "loss_curve.png"), dpi=120)
plt.show()
print(f"Perdida final: {losses[-1]:.5f}")